In [1]:
from retriever_v2.base import Document
from retriever_v2.bm25 import BM25Retriever
from retriever_v2.utils import INDEX_DIR

import pandas as pd

In [2]:
docs = [
    Document(file.stem.split("_")[0], file.read_text(encoding="utf-8"))
    for file in (INDEX_DIR / "docs").glob("*.txt")
]

In [3]:
retriever = BM25Retriever(documents=docs)

Tokenizing documents: 100%|██████████| 12976/12976 [01:22<00:00, 156.91it/s]


In [4]:
test_queries = (
    pd.read_csv(INDEX_DIR / "test_scores.csv")
    .rename(columns={"Unnamed: 0": "doc_id"})
    .set_index("doc_id")
)

In [5]:
from sklearn.metrics import ndcg_score

In [6]:
res = {}

for query in test_queries.columns:
    scores_df = pd.DataFrame(retriever.score(query)).set_index("doc_id")
    scores_df["gold_standard"] = test_queries[query][scores_df.index]
    score = ndcg_score(
        [scores_df["gold_standard"].values], [scores_df["score"].values]
    )
    res[query] = score

In [7]:
res_df = pd.Series(res)

In [8]:
res_df.mean()

0.7046193012798269